# SmartGlove: evidence-led data quality audit

**Protocol:** 10 participants × 10 gestures × 5 trials × 50 samples. No final classifier is trained.

## Start here on Kaggle

1. Create a Python notebook and attach your ZIP as a private dataset, or attach the extracted dataset. Keep participant data private unless you have permission to share it.
2. Import this `.ipynb` notebook into the notebook editor. A CPU session is sufficient; no GPU or internet is required to run it.
3. Run the cells from top to bottom. Automatic discovery searches `/kaggle/input`. If multiple candidate datasets are found, set `DATA_HINT` to the exact ZIP or extracted root you want. The notebook intentionally refuses to guess between copies.
4. Check the discovery table, then the hard audit and accepted-log join. A hard failure stops downstream interpretation and leaves an audit file.
5. Read the generated final narrative and dashboard, then inspect the QC examples. Download `/kaggle/working/EDA_results` after the run.

**Statistical units:** 500 trial records, clustered within **10 participants**. Calling a trial an experimental unit does not make five repetitions from the same person statistically independent. This notebook uses descriptive summaries; it does not use 25,000 samples as a statistical sample size, nor give population inference treating 500 trials as independent people. Later validation must hold out whole participants, with model selection inside training-participant folds.

**Threshold policy:** structural rules come from the supplied protocol; ADC bounds from its stated 12-bit format. Distributional fences are explicitly exploratory. We do not delete flagged trials. Logged 5°/50°/s limits apply only where the logs actually state calibration/fist criteria, not as invented universal gesture cutoffs.

**Hardware caveat:** conversions use your stated firmware configuration (16.4 LSB per °/s, 8192 LSB per g). The archive does not include firmware/register dumps, so these configuration values cannot be independently established from CSVs. Original values are preserved.

**Audit conducted on the uploaded archive:** the opening cells compute all counts afresh. Do not substitute the expected figures for observed results.


In [ ]:
from pathlib import Path
import os, re, json, zipfile, hashlib, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
try:
    from IPython.display import display, Markdown
except ImportError:
    def display(x): print(x.to_string(index=False) if isinstance(x, pd.DataFrame) else x)
    def Markdown(x): return x

# Leave these alone on Kaggle. Environment overrides support local reproducibility.
INPUT = Path(os.environ.get('SMARTGLOVE_INPUT', '/kaggle/input'))
OUT = Path(os.environ.get('SMARTGLOVE_OUTPUT', '/kaggle/working/EDA_results'))
DATA_HINT = None  # Optional exact path to the intended ZIP or extracted root.
OUT.mkdir(parents=True, exist_ok=True)
PLOTS = OUT / 'plots'; PLOTS.mkdir(exist_ok=True)
FLEX = ['flex_thumb','flex_index','flex_middle','flex_ring','flex_little']
Q = ['qw','qx','qy','qz']
SCHEMA = ['timestamp_ms','trial_id'] + FLEX + Q + ['gx','gy','gz','ax','ay','az','roll','pitch','yaw']
KEY = ['participant','gesture_id','trial_id']
PARTS = [f'P{i:02d}' for i in range(1,11)]
GEST = [f'G{i:02d}' for i in range(1,11)]
NAMES = ['Fist','Excellent','Stop','Thumbs Up','This Way','Wait','Hey You!','Victory','Call Me','Attention']
NAME_MAP = dict(zip(GEST,NAMES))
pd.set_option('display.max_rows', 25)
pd.set_option('display.max_columns', 14)
plt.rcParams.update({'figure.dpi':110,'font.size':10})
OUTPUT_DESCRIPTIONS = {}
def save(df, name, meaning):
    df.to_csv(OUT/name, index=False)
    OUTPUT_DESCRIPTIONS[name]=meaning
    return df

def plot_save(name):
    plt.tight_layout()
    plt.savefig(PLOTS/f'{name}.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

def heatmap(df, title, name, annotate=False):
    fig,ax=plt.subplots(figsize=(max(7,len(df.columns)*.65), max(4,len(df)*.35)))
    im=ax.imshow(df.to_numpy(float),aspect='auto',cmap='viridis')
    ax.set_xticks(range(len(df.columns)),df.columns,rotation=45,ha='right')
    ax.set_yticks(range(len(df.index)),df.index)
    ax.set_title(title); fig.colorbar(im,ax=ax)
    if annotate:
        for i in range(len(df)):
            for j in range(len(df.columns)): ax.text(j,i,f'{df.iloc[i,j]:g}',ha='center',va='center',color='white')
    plot_save(name)

def q_unit(q):
    q=np.asarray(q,float); norm=np.linalg.norm(q,axis=-1,keepdims=True)
    if not np.isfinite(q).all() or np.any(norm==0): raise ValueError('Invalid zero/nonfinite quaternion; inspect source.')
    return q/norm

def q_angle(q,r):
    return np.degrees(2*np.arccos(np.clip(np.abs(np.sum(q_unit(q)*q_unit(r),axis=-1)),0,1)))

def q_mean(q):
    u=q_unit(q)
    # Markley mean: dominant eigenvector of sum(q q^T); invariant to q -> -q.
    w,v=np.linalg.eigh(u.T@u)
    return v[:,-1]

def q_metrics(frame):
    u=q_unit(frame[Q]); mean=q_mean(u); ang=q_angle(u,mean)
    # Exact pairwise diameter is affordable for 50 samples.
    diameter=np.degrees(2*np.arccos(np.clip(np.abs(u@u.T),0,1))).max()
    return {'orientation_p95_deg':np.percentile(ang,95),'orientation_max_from_mean_deg':ang.max(),
            'orientation_diameter_deg':diameter,'orientation_end_to_start_deg':q_angle(u[-1],u[0]),
            'q_norm_error_max':np.abs(np.linalg.norm(frame[Q],axis=1)-1).max()}

def units(df):
    d=df.copy()
    for c in ['gx','gy','gz']: d[c+'_dps']=d[c]/16.4
    for c in ['ax','ay','az']: d[c+'_g']=d[c]/8192
    d['gyro_magnitude_dps']=np.linalg.norm(d[['gx_dps','gy_dps','gz_dps']],axis=1)
    d['accel_magnitude_g']=np.linalg.norm(d[['ax_g','ay_g','az_g']],axis=1)
    return d

print('Python',platform.python_version(), '| numpy',np.__version__,'| pandas',pd.__version__)


## Phase 0 — Discover the actual dataset

**A. Purpose.** Locate the archive or extracted root and enumerate every file before interpreting signals.

**C. Expected output.** A compact participant table, schema inventory, accepted counts, session table, unexpected-file list and full manifest on disk.

**D. Interpretation.** Gesture data, calibration, checkpoints and rejected recordings serve different purposes; only G01–G10 form accepted data.

**E. Genuine problem.** Missing dataset, multiple candidate roots, unreadable CSVs or unknown gesture files require investigation.

**F. Warning only.** Extra checkpoints, unused calibration sessions and duplicate metadata are investigated separately.

**G. Do not conclude.** A filename or declared participant completion does not prove valid data.

**B. Runnable code:**


In [ ]:
def looks_like_root(p): return (p/'raw').is_dir() and (p/'gesture_map.csv').exists()
if DATA_HINT:
    source=Path(DATA_HINT)
else:
    roots=sorted({p.parent for p in INPUT.rglob('gesture_map.csv') if looks_like_root(p.parent)})
    archives=[]
    for z in INPUT.rglob('*.zip'):
        with zipfile.ZipFile(z) as f:
            if any(n.endswith('gesture_map.csv') for n in f.namelist()): archives.append(z)
    choices=roots+archives
    if len(choices)!=1:
        raise RuntimeError(f'Found {len(choices)} candidate datasets: {choices}. Set DATA_HINT explicitly.')
    source=choices[0]
if source.suffix.lower()=='.zip':
    digest=hashlib.sha256(source.read_bytes()).hexdigest()
    dest=OUT/'extracted_input'/digest[:12]; dest.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(source) as z:
        for member in z.infolist():
            target=(dest/member.filename).resolve()
            if not target.is_relative_to(dest.resolve()): raise ValueError('Unsafe archive path')
            if (member.external_attr>>16)&0o170000 == 0o120000: raise ValueError('Archive symlink rejected')
        z.extractall(dest)
    roots=[p.parent for p in dest.rglob('gesture_map.csv') if looks_like_root(p.parent)]
    if len(roots)!=1: raise RuntimeError(f'Ambiguous extracted roots: {roots}')
    ROOT=roots[0]
else:
    ROOT=source; digest=None
P_DIRS=sorted(p for p in (ROOT/'raw').iterdir() if p.is_dir())
manifest=[]; schema_inventory=[]; cache={}
for p in sorted(ROOT.rglob('*')):
    if not p.is_file(): continue
    record={'path':str(p.relative_to(ROOT)), 'bytes':p.stat().st_size,
            'sha256':hashlib.sha256(p.read_bytes()).hexdigest()}
    if p.suffix.lower()=='.csv':
        d=pd.read_csv(p); cache[p]=d
        record['rows']=len(d)
        schema_inventory.append({'file':record['path'],'rows':len(d),'columns':' | '.join(d.columns)})
    manifest.append(record)
save(pd.DataFrame(manifest),'file_manifest.csv','All source paths, row counts, byte sizes and SHA256 digests; raw sources are not modified.')
save(pd.DataFrame(schema_inventory),'schema_inventory.csv','Exact observed schema for every CSV, including support files.')
structure=[]; unknown=[]
for p in P_DIRS:
    names=[x.name for x in p.glob('*.csv')]
    sessions=sorted(set(s for name in names for s in re.findall(r'S\d+',name)))
    structure.append(dict(participant=p.name,n_gesture_files=len(list(p.glob('G[0-9][0-9].csv'))),
        n_calibration_files=len(list(p.glob('calibration_S*.csv'))), n_fist_files=len(list(p.glob('fist_S*.csv'))),
        n_checkpoints=len(list(p.glob('checkpoint_*.csv'))),n_drift_files=len(list(p.glob('drift_check_*.csv'))),
        n_rejected_files=len(list(p.glob('rejected/*.csv'))),sessions_detected=','.join(sessions)))
    for f in p.rglob('*'):
        if not f.is_file(): continue
        known=(re.fullmatch(r'(G\d{2}|calibration_S\d+|fist_S\d+|checkpoint_S\d+_R\d+|drift_check_S\d+)\.csv',f.name)
               or f.name in ['calibration_summary.csv','references.csv','order.csv','trial_log.csv','session_log.csv']
               or (f.parent.name=='rejected' and re.fullmatch(r'G\d+_T\d+_A\d+_DISCARDED\.csv',f.name)))
        if not known: unknown.append(str(f.relative_to(ROOT)))
for f in ROOT.iterdir():
    if f.name not in ['raw','pilot','participants.csv','gesture_map.csv','gesture_definitions.csv']: unknown.append(f.name)
structure=pd.DataFrame(structure)
save(structure,'participant_structure.csv','Compact structural inventory by participant.')
print(f'{ROOT.name}/\n  gesture_definitions.csv, gesture_map.csv, participants.csv\n  pilot/\n  raw/ [participant table below]')
display(structure)
print('Unknown files:',unknown)
print('Pilot files:',[str(p.relative_to(ROOT)) for p in (ROOT/'pilot').rglob('*') if p.is_file()])
GFILES=sorted((ROOT/'raw').glob('*/G[0-9][0-9].csv'))
raw=pd.concat([cache[p].assign(participant=p.parent.name,gesture_id=p.stem,data_file=str(p.relative_to(ROOT))) for p in GFILES],ignore_index=True)
logs=pd.concat([cache[p/'trial_log.csv'].assign(participant=p.name,log_row=np.arange(len(cache[p/'trial_log.csv']))+2) for p in P_DIRS],ignore_index=True)
events=pd.concat([cache[p/'session_log.csv'].assign(participant=p.name) for p in P_DIRS],ignore_index=True)
accepted_logs=logs.loc[logs.result.eq('ACCEPTED')].copy()
print('Participants:',sorted(raw.participant.unique()))
print('Gestures:',sorted(raw.gesture_id.unique()))
print('Accepted trials in CSV:',len(raw[KEY].drop_duplicates()),'| accepted log rows:',len(accepted_logs),'| sample rows:',len(raw))
print('Gesture schemas:',{tuple(cache[p].columns) for p in GFILES})
print('Supporting schema variants:')
display(pd.DataFrame(schema_inventory).groupby('columns',sort=False).agg(n_files=('file','size'),example=('file','first')).reset_index())
print('Accepted trials by session (empty calibration sessions appear later):')
display(pd.crosstab(accepted_logs.participant,accepted_logs.session))
print('Core count agreement with protocol:', len(P_DIRS)==10 and len(GFILES)==100 and len(raw)==25000 and len(raw[KEY].drop_duplicates())==500)


## Phase 1 — Hard structural integrity audit

**A. Purpose.** Check objective collection rules without silently repairing the input.

**C. Expected output.** A PASS/WARNING/FAIL audit plus per-trial integrity records. All expected counts are computed.

**D. Interpretation.** 50 timestamps from 0 to 1960 ms describe 50 samples of a nominal 2-second record; the first-to-last interval is 1.96 seconds.

**E. Genuine problem.** Missing, nonnumeric, nonfinite values; wrong schema/count; duplicate rows within their recording context; invalid timestamps.

**F. Warning only.** Identical values in different participants are not necessarily duplicate records. Metadata duplicates are warnings until they affect identity.

**G. Do not conclude.** Perfect timestamps do not independently prove physical sample timing: firmware may write a scheduled time grid.

**B. Runnable code:**


In [ ]:
audit=[]
def check(name,expected,observed,ok,details='',warning=False):
    audit.append(dict(check=name,expected=str(expected),observed=str(observed),status='PASS' if ok else ('WARNING' if warning else 'FAIL'),details=details))
def flush_audit():
    return save(pd.DataFrame(audit),'structure_audit.csv','Formal structural and provenance checks; FAIL is reserved for hard inconsistencies.')
check('participant folders',PARTS,[p.name for p in P_DIRS],[p.name for p in P_DIRS]==PARTS)
check('gesture files',100,len(GFILES),len(GFILES)==100)
for p in P_DIRS:
    actual=sorted(f.stem for f in p.glob('G[0-9][0-9].csv'))
    check(p.name+' gesture IDs',GEST,actual,actual==GEST)
file_issues=[]
for p in GFILES:
    d=cache[p]
    if list(d.columns)!=SCHEMA: file_issues.append((str(p.relative_to(ROOT)),'schema'))
    if len(d)!=250: file_issues.append((str(p.relative_to(ROOT)),'rows'))
    if 'trial_id' not in d or set(d.trial_id)!=set(range(1,6)): file_issues.append((str(p.relative_to(ROOT)),'trial IDs'))
check('100 file schemas, lengths and trial IDs','20 exact columns; 250 rows; trial IDs 1–5',file_issues,not file_issues)
if any(x[1]=='schema' for x in file_issues):
    display(flush_audit()); raise RuntimeError('Schema mismatch; inspect structure_audit.csv before proceeding.')
numeric=raw[SCHEMA].apply(pd.to_numeric,errors='coerce')
parse_errors=int((numeric.isna() & raw[SCHEMA].notna()).sum().sum())
check('numeric parsing',0,parse_errors,parse_errors==0)
check('missing source values',0,int(raw[SCHEMA].isna().sum().sum()),not raw[SCHEMA].isna().any().any())
check('infinite values',0,int(np.isinf(numeric).sum().sum()),not np.isinf(numeric).any().any())
raw[SCHEMA]=numeric
integrity=[]
for key,d in raw.groupby(KEY,sort=True,dropna=False):
    t=d.timestamp_ms.to_numpy(); duplicate=int(d[SCHEMA].duplicated().sum())
    integrity.append(dict(zip(KEY,key),n_rows=len(d),missing=int(d[SCHEMA].isna().sum().sum()),
       infinite=int(np.isinf(d[SCHEMA]).sum().sum()),duplicate_rows=duplicate,
       timestamp_ok=np.array_equal(t,np.arange(50)*40),
       timestamp_strict=bool(np.all(np.diff(t)>0)), timestamp_duplicates=int(d.timestamp_ms.duplicated().sum())))
integrity=pd.DataFrame(integrity)
save(integrity,'trial_integrity.csv','Objective row, timestamp, missing and duplicate checks for each trial.')
check('accepted trial count',500,len(integrity),len(integrity)==500)
check('accepted sample count',25000,len(raw),len(raw)==25000)
check('rows/trial',50,integrity.n_rows.value_counts().to_dict(),integrity.n_rows.eq(50).all())
check('timestamp schedule','0:40:1960',int((~integrity.timestamp_ok).sum()),integrity.timestamp_ok.all())
check('strict increasing timestamps',True,integrity.timestamp_strict.all(),integrity.timestamp_strict.all())
check('duplicate rows within trial',0,int(integrity.duplicate_rows.sum()),integrity.duplicate_rows.sum()==0)
# Full-payload equality across files is reported separately, never counted as proof of corruption.
check('identical rows across all files (context omitted)',0,int(raw[SCHEMA].duplicated().sum()),not raw[SCHEMA].duplicated().any(),warning=True)
participants=cache[ROOT/'participants.csv']
duplicate_metadata=participants[participants.duplicated('participant_id',keep=False)]
check('unique participant metadata','one row/ID',duplicate_metadata.participant_id.unique().tolist(),duplicate_metadata.empty,warning=True)
check('metadata participant set',PARTS,sorted(participants.participant_id.unique()),sorted(participants.participant_id.unique())==PARTS)
defs=cache[ROOT/'gesture_definitions.csv']; gm=cache[ROOT/'gesture_map.csv']
check('gesture map IDs',GEST,sorted(gm.gesture_id),sorted(gm.gesture_id)==GEST)
check('gesture map names',NAME_MAP,dict(zip(gm.gesture_id,gm.gesture_name)),dict(zip(gm.gesture_id,gm.gesture_name))==NAME_MAP)
check('completed gesture descriptions','no placeholders',int(defs.description.str.contains('fill in',case=False,na=False).sum()),not defs.description.str.contains('fill in',case=False,na=False).any(),warning=True)
check('unknown files','none',unknown,not unknown,warning=True)
display(flush_audit())
if any(a['status']=='FAIL' for a in audit):
    save(integrity.loc[(integrity.n_rows!=50)|(~integrity.timestamp_ok)|(integrity.missing>0)|(integrity.infinite>0)|(integrity.duplicate_rows>0)],
         'hard_integrity_failures.csv','Repair provenance or acquisition errors before feature extraction.')
    raise RuntimeError('Hard integrity failures: downstream EDA stopped; raw inputs retained.')


## Phase 2 — Join accepted trials to their exact log records

**A. Purpose.** Build a one-to-one trial index and verify references, boot/epoch, planned order and rejected-file separation.

**C. Expected output.** 500 mapped records with session, round, attempt, reference and logged QC values.

**D. Interpretation.** A failed/discarded attempt can share participant/gesture/trial ID with the later accepted retry. Attempt identity is distinct.

**E. Genuine problem.** Ambiguous accepted mapping, missing accepted log, or mismatch in reference/boot/epoch must be resolved before downstream interpretation.

**F. Warning only.** No CSV for a failed connection is expected if recording never completed.

**G. Do not conclude.** Without video or per-sample attempt IDs, absence of rejected payload copies cannot prove semantic gesture correctness.

**B. Runnable code:**


In [ ]:
index=raw.groupby(KEY,as_index=False).agg(data_file=('data_file','first'),n_samples=('timestamp_ms','size'))
dup=accepted_logs.duplicated(KEY,keep=False)
check('one ACCEPTED row/key',0,int(dup.sum()),not dup.any(),accepted_logs.loc[dup,KEY].to_json(orient='records'))
if dup.any(): display(accepted_logs.loc[dup]); display(flush_audit()); raise RuntimeError('Ambiguous accepted mapping; no guessing.')
coverage=index[KEY].merge(accepted_logs[KEY],on=KEY,how='outer',indicator=True,validate='one_to_one')
check('accepted CSV/log key coverage','both for every key',coverage._merge.value_counts().to_dict(),coverage._merge.eq('both').all())
if not coverage._merge.eq('both').all(): display(coverage[coverage._merge!='both']); display(flush_audit()); raise RuntimeError('CSV/log mismatch')
index=index.merge(accepted_logs.rename(columns={'data_file':'logged_data_file'}),on=KEY,validate='one_to_one')
check('accepted data file agrees',True, bool((index.logged_data_file==index.gesture_id+'.csv').all()),(index.logged_data_file==index.gesture_id+'.csv').all())
check('gesture names agree',True,bool(index.gesture_name.eq(index.gesture_id.map(NAME_MAP)).all()),index.gesture_name.eq(index.gesture_id.map(NAME_MAP)).all())
refs=pd.concat([cache[p/'references.csv'].assign(participant=p.name) for p in P_DIRS],ignore_index=True)
calmeta=pd.concat([cache[p/'calibration_summary.csv'].assign(participant=p.name) for p in P_DIRS],ignore_index=True)
RKEY=['participant','session','ref_id']
check('unique session/reference keys',0,int(refs.duplicated(RKEY).sum()),not refs.duplicated(RKEY).any())
check('unique session calibration keys',0,int(calmeta.duplicated(RKEY).sum()),not calmeta.duplicated(RKEY).any())
if refs.duplicated(RKEY).any() or calmeta.duplicated(RKEY).any(): display(flush_audit()); raise RuntimeError('Ambiguous reference/calibration')
refmap=index.merge(refs[RKEY+['boot','epoch']],on=RKEY,how='left',suffixes=('','_ref'),validate='many_to_one',indicator=True)
refbad=(refmap._merge!='both')|refmap.boot.ne(refmap.boot_ref)|refmap.epoch.ne(refmap.epoch_ref)
check('reference identity and boot/epoch',0,int(refbad.sum()),not refbad.any())
orders=pd.concat([cache[p/'order.csv'].assign(participant=p.name) for p in P_DIRS],ignore_index=True)
orderkey=['participant','round','position']
check('unique planned order slot',0,int(orders.duplicated(orderkey).sum()),not orders.duplicated(orderkey).any())
if not orders.duplicated(orderkey).any():
    ordermatch=index.merge(orders[orderkey+['gesture_id']],on=orderkey,how='left',suffixes=('','_planned'),validate='many_to_one')
    check('accepted order matches plan',0,int(ordermatch.gesture_id.ne(ordermatch.gesture_id_planned).sum()),ordermatch.gesture_id.eq(ordermatch.gesture_id_planned).all())
# Match rejected CSV paths and complete ordered payloads independently.
rejected_audit=[]
for p in sorted((ROOT/'raw').glob('*/rejected/*.csv')):
    pid=p.parent.parent.name; rel=str(p.relative_to(p.parent.parent))
    log=logs[(logs.participant==pid)&(logs.data_file==rel)&(logs.result=='DISCARDED')]
    match=re.fullmatch(r'(G\d+)_T(\d+)_A(\d+)_DISCARDED\.csv',p.name)
    payload=cache[p]; same=[]
    if match:
        for key,d in raw[(raw.participant==pid)&(raw.gesture_id==match[1])].groupby(KEY):
            if payload[SCHEMA].reset_index(drop=True).equals(d[SCHEMA].reset_index(drop=True)): same.append(str(key))
    identity_ok=(len(log)==1 and match is not None and log.iloc[0].gesture_id==match[1]
                 and int(log.iloc[0].trial_id)==int(match[2]) and int(log.iloc[0].attempt)==int(match[3]))
    rejected_audit.append({'participant':pid,'file':rel,'rows':len(payload),'matching_discard_events':len(log),
                           'identity_ok':identity_ok,'accepted_payload_matches':' | '.join(same)})
rejected_audit=pd.DataFrame(rejected_audit)
check('rejected files match discarded events',True,bool(rejected_audit.identity_ok.all()),rejected_audit.identity_ok.all())
check('rejected payload copied into accepted data',0,int(rejected_audit.accepted_payload_matches.ne('').sum()),rejected_audit.accepted_payload_matches.eq('').all())
missing_discard=[]
for row in logs[logs.result=='DISCARDED'].itertuples():
    if pd.isna(row.data_file) or not (ROOT/'raw'/row.participant/str(row.data_file)).is_file(): missing_discard.append(f'{row.participant}/{row.gesture_id}/T{row.trial_id}/A{row.attempt}')
check('discarded log recording exists',0,missing_discard,not missing_discard)
save(rejected_audit,'rejected_file_audit.csv','Rejected-file event identity and exact accepted-payload contamination check.')
save(index,'accepted_trial_index.csv','One row per accepted participant/gesture/trial; complete logs and verified session/reference mapping.')
attempt_summary=pd.crosstab(logs.participant,logs.result).reindex(columns=['ACCEPTED','DISCARDED','FAILED'],fill_value=0).reset_index()
save(attempt_summary,'attempt_summary.csv','Attempt counts, including retries; these are not additional independent accepted trials.')
display(attempt_summary); display(index[KEY+['session','round','position','attempt','ref_id','boot','epoch']].head(12))
display(flush_audit())
if any(a['status']=='FAIL' for a in audit): raise RuntimeError('Provenance inconsistency: resolve audit FAIL before downstream analysis.')


## Phase 3 — Experimental events and session segmentation

**A. Purpose.** Separate metadata irregularities and collection events from defects in accepted signals.

**C. Expected output.** Participant summaries, session summaries, reason counts and an event inventory.

**D. Interpretation.** Restarts create new reference chains. Calibration-only sessions are retained with zero accepted trials.

**E. Genuine problem.** Using a reference from the wrong session would compromise orientation interpretation.

**F. Warning only.** An extra R06 checkpoint or missing completion event need documentation, not automatic participant exclusion.

**G. Do not conclude.** Link-loss events are not equivalent to failed trials, and a final drift in an unused session does not validate earlier sessions.

**B. Runnable code:**


In [ ]:
sessions=[]
for p in P_DIRS:
    ev=events[events.participant==p.name]; tl=logs[logs.participant==p.name]
    available=sorted(set(ev.session.dropna())|set(refs.loc[refs.participant==p.name,'session']))
    for session in [x for x in available if re.fullmatch(r'S\d+',str(x))]:
        e=ev[ev.session==session]; a=tl[(tl.session==session)&(tl.result=='ACCEPTED')]
        sessions.append(dict(participant=p.name,session=session,accepted_trials=len(a),
          accepted_calibrations=int(e.event.eq('CALIBRATION_ACCEPTED').sum()),
          rejected_calibrations=int(e.event.eq('CAL_REJECTED').sum()),fist_references=int(e.event.eq('FIST_REFERENCE').sum()),
          link_losses=int(e.event.eq('LINK_LOST').sum()),session_start=int(e.event.eq('SESSION_START').sum()),
          recalibration_needed=int(e.event.eq('RECALIBRATION_NEEDED').sum()),
          participant_complete=e.event.eq('PARTICIPANT_COMPLETE').any(),drift_event=e.event.eq('DRIFT_CHECK').any(),
          drift_file=(p/f'drift_check_{session}.csv').exists()))
session_summary=pd.DataFrame(sessions)
save(session_summary,'session_summary.csv','All named sessions, including zero-accepted-trial sessions; events with session - remain in event_inventory.csv.')
save(events,'event_inventory.csv','Full session-event evidence; details preserve recorded firmware criteria and restart explanations.')
participant_events=[]
for p in P_DIRS:
    e=events[events.participant==p.name]; ss=session_summary[session_summary.participant==p.name]
    row={'participant':p.name,'sessions':len(ss),'accepted_sessions':int(ss.accepted_trials.gt(0).sum()),
         'final_drift_files':len(list(p.glob('drift_check_*.csv')))}
    for kind in ['CALIBRATION_ACCEPTED','CAL_REJECTED','FIST_REFERENCE','LINK_LOST','LINK_RESTORED','SESSION_START','RECALIBRATION_NEEDED','PARTICIPANT_COMPLETE','PAUSED']:
        row[kind]=int(e.event.eq(kind).sum())
    participant_events.append(row)
participant_events=pd.DataFrame(participant_events).merge(attempt_summary,on='participant')
save(participant_events,'participant_event_summary.csv','Participant-level counts include unnamed-session calibration rejections and link events.')
display(participant_events)
display(session_summary[['participant','session','accepted_trials','drift_event','participant_complete']])
reasons=logs[logs.result!='ACCEPTED'].groupby(['result','reason'],dropna=False).size().reset_index(name='count')
save(reasons,'attempt_reasons.csv','Unsuccessful attempts by result and operator/connection reason.'); display(reasons)
irregularities=[]
for pid in duplicate_metadata.participant_id.unique():
    irregularities.append({'participant':pid,'category':'METADATA IRREGULARITY','finding':'Repeated participant metadata ID; creation times differ. Resolve history without duplicating signal data.'})
for p in P_DIRS:
    if not (events.participant.eq(p.name)&events.event.eq('PARTICIPANT_COMPLETE')).any():
        irregularities.append({'participant':p.name,'category':'PROTOCOL/SESSION EVENT','finding':'No PARTICIPANT_COMPLETE event; accepted CSV count independently audited.'})
    if not list(p.glob('drift_check_*.csv')):
        irregularities.append({'participant':p.name,'category':'PROTOCOL/SESSION EVENT','finding':'No final drift CSV; keep checkpoint evidence separately.'})
    for f in p.glob('checkpoint_*.csv'):
        rnd=int(re.search(r'_R(\d+)',f.name)[1])
        if rnd>5: irregularities.append({'participant':p.name,'category':'PROTOCOL/SESSION EVENT','finding':f'Extra checkpoint {f.name}; may be an endpoint checkpoint, not a sixth gesture repetition.'})
for row in session_summary.itertuples():
    if row.drift_event and row.accepted_trials==0:
        irregularities.append({'participant':row.participant,'category':'PROTOCOL/SESSION EVENT','finding':f'Final drift in {row.session}, a session with zero accepted trials; not evidence of earlier-session drift.'})
irregularities.append({'participant':'ALL','category':'METADATA IRREGULARITY','finding':'Gesture descriptions contain placeholders; document exact hand shape and posture before publication.'})
irregularities=pd.DataFrame(irregularities)
save(irregularities,'irregularities.csv','Documented anomalies classified separately from signal defects.'); display(irregularities)


## Phase 4 — Class balance at trial level

**A. Purpose.** Confirm every participant/gesture cell contains five trials.

**C. Expected output.** Counts of 50 per participant and gesture, and a 10 × 10 table containing fives.

**D. Interpretation.** Balanced accepted counts support fair descriptive comparisons.

**E. Genuine problem.** Missing cells or unexpected counts would violate the final design.

**F. Warning only.** Equal numbers do not guarantee equally representative gestures or people.

**G. Do not conclude.** Balance does not establish generalization to unseen participants.

**B. Runnable code:**


In [ ]:
balance=pd.crosstab(index.participant,index.gesture_id).reindex(index=PARTS,columns=GEST,fill_value=0)
save(balance.reset_index(),'trial_balance.csv','Accepted trial counts, never sample rows.')
display(pd.DataFrame({'participant_trials':balance.sum(axis=1)}))
display(pd.DataFrame({'gesture_trials':balance.sum(axis=0)}))
heatmap(balance,'Accepted trials per participant and gesture','trial_balance',True)
assert balance.eq(5).all().all()


## Phase 5 — Raw flex sensor quality

**A. Purpose.** Quantify range, central tendency, variability, rails, flat channels and steps without filtering.

**C. Expected output.** Nine statistics per finger per trial; distributions and participant/gesture profiles.

**D. Interpretation.** Large range or adjacent steps identify review candidates. A constant channel in a static hold can reflect quantization, not a broken sensor.

**E. Genuine problem.** Values outside 0–4095 violate the stated ADC encoding; exact rail readings need inspection for clipping.

**F. Warning only.** Near-rail 1% margins and later IQR fences are heuristics, not validated acceptance thresholds.

**G. Do not conclude.** Different ADC levels do not directly measure bend angles or prove gesture mislabeling.

**B. Runnable code:**


In [ ]:
feature_rows=[]
for key,d in raw.groupby(KEY,sort=True):
    row=dict(zip(KEY,key))
    for c in FLEX:
        v=d[c].to_numpy()
        for label,val in {'median':np.median(v),'mean':np.mean(v),'std':np.std(v,ddof=1),
           'iqr':np.percentile(v,75)-np.percentile(v,25),'min':v.min(),'max':v.max(),
           'range':np.ptp(v),'p05':np.percentile(v,5),'p95':np.percentile(v,95)}.items(): row[c+'_'+label]=val
        row[c+'_rail_samples']=int(((v==0)|(v==4095)).sum())
        row[c+'_outside_samples']=int(((v<0)|(v>4095)).sum())
        row[c+'_near_rail_samples_heuristic']=int(((v<=.01*4095)|(v>=.99*4095)).sum())
        row[c+'_unique_values']=len(np.unique(v))
        row[c+'_max_step']=np.abs(np.diff(v)).max()
    feature_rows.append(row)
flex_features=pd.DataFrame(feature_rows)
T=index.merge(flex_features,on=KEY,validate='one_to_one')
save(flex_features,'flex_trial_statistics.csv','All nine requested flex statistics plus rails, distinct values and largest adjacent step.')
rail_summary=pd.DataFrame([{'finger':c,'min':raw[c].min(),'max':raw[c].max(),
   'exact_rail_samples':int(((raw[c]==0)|(raw[c]==4095)).sum()),
   'outside_samples':int(((raw[c]<0)|(raw[c]>4095)).sum()),
   'constant_trials':int(T[c+'_unique_values'].eq(1).sum())} for c in FLEX])
save(rail_summary,'flex_sensor_summary.csv','Sample-level physical range and trial-level constant-channel counts.'); display(rail_summary)
fig,axes=plt.subplots(1,5,figsize=(16,3))
for ax,c in zip(axes,FLEX): ax.hist(raw[c],bins=40); ax.set_title(c.replace('flex_','')); ax.set_xlabel('Raw ADC'); ax.set_ylabel('Samples (descriptive)')
plot_save('flex_sample_distributions')
for grouping in ['gesture_id','participant']:
    fig,axes=plt.subplots(2,3,figsize=(15,8))
    for ax,c in zip(axes.flat,FLEX):
        labels=sorted(T[grouping].unique()); ax.boxplot([T.loc[T[grouping]==g,c+'_median'] for g in labels],tick_labels=labels)
        ax.set_title(c+' trial median'); ax.tick_params(axis='x',rotation=45); ax.set_ylabel('ADC')
    axes.flat[-1].axis('off'); plot_save('flex_medians_by_'+grouping)
fig,ax=plt.subplots(figsize=(9,4)); ax.boxplot([T[c+'_range'] for c in FLEX],tick_labels=FLEX); ax.set_ylabel('Within-trial range, ADC'); plot_save('flex_trial_ranges')


## Phase 6 — Session calibration and fist dynamic range

**A. Purpose.** Compare each session’s neutral calibration with its own fist reference and recorded summary.

**C. Expected output.** A session/finger table with signed spans, variability, raw gyro bias, acceleration and quaternion checks.

**D. Interpretation.** Signed spans determine empirical bending direction. Noise-to-span ratio identifies fragile scaling denominators.

**E. Genuine problem.** Missing or ambiguous session references, nonfinite/zero quaternion, or zero span prevent that calculation.

**F. Warning only.** Calibration summary means and recomputed rounded CSV values may differ slightly. The gyro summary may use a separate firmware acquisition; compare, do not assume equality.

**G. Do not conclude.** A stable neutral hold does not guarantee correct absolute orientation or constant glove fit.

**B. Runnable code:**


In [ ]:
calrows=[]
for r in calmeta.itertuples(index=False):
    p=ROOT/'raw'/r.participant; f=p/f'calibration_{r.session}.csv'; ff=p/f'fist_{r.session}.csv'
    if not f.exists(): raise RuntimeError(f'Missing calibration file {f}')
    d=units(cache[f]); fist=cache.get(ff)
    meta=calmeta[(calmeta.participant==r.participant)&(calmeta.session==r.session)&(calmeta.ref_id==r.ref_id)]
    same=refs[(refs.participant==r.participant)&(refs.session==r.session)&(refs.ref_id==r.ref_id)]
    if len(meta)!=1 or len(same)!=1: raise RuntimeError('Ambiguous calibration reference')
    qm=q_metrics(d)
    for c in FLEX:
        neutral=d[c].mean(); fm=fist[c].median() if fist is not None else np.nan
        span=fm-neutral
        row={'participant':r.participant,'session':r.session,'ref_id':r.ref_id,'finger':c,
             'neutral_mean':neutral,'neutral_std':d[c].std(),'neutral_iqr':d[c].quantile(.75)-d[c].quantile(.25),
             'recorded_neutral_mean':meta.iloc[0][c+'_mean'],'neutral_mean_difference':neutral-meta.iloc[0][c+'_mean'],
             'fist_median':fm,'fist_std':fist[c].std() if fist is not None else np.nan,'signed_span':span,'absolute_span':abs(span),
             'noise_span_ratio':max(d[c].std(),fist[c].std())/abs(span) if fist is not None and span!=0 else np.nan,
             'calibration_rows':len(d),'fist_rows':len(fist) if fist is not None else 0,
             'calibration_temp_c':r.temp_c,'logged_spread_deg':r.spread_deg,'logged_max_dps':r.max_dps,
             'recomputed_gyro_peak_dps':d.gyro_magnitude_dps.max(),
             'reference_to_recomputed_mean_deg':float(q_angle(q_mean(d[Q]),same[Q].to_numpy()[0])),**qm}
        for axis in ['x','y','z']:
            row[f'recomputed_gyro_bias_{axis}_dps']=d[f'g{axis}_dps'].mean()
            row[f'logged_gyro_bias_{axis}_dps']=meta.iloc[0][f'gyro_bias_{axis}_dps']
            row[f'recomputed_accel_{axis}_mean_g']=d[f'a{axis}_g'].mean()
            row[f'logged_accel_{axis}_mean_g']=meta.iloc[0][f'accel_{axis}_mean_g']
        calrows.append(row)
cal=pd.DataFrame(calrows)
save(cal,'calibration_summary_eda.csv','Session/finger neutral and fist statistics, signed spans, summary cross-checks and stability metrics.')
display(cal.groupby('finger').agg(min_span=('absolute_span','min'),median_span=('absolute_span','median'),max_span=('absolute_span','max'),min_signed=('signed_span','min'),max_signed=('signed_span','max'),max_noise_span=('noise_span_ratio','max')))
display(cal[['participant','session','finger','signed_span','neutral_std','noise_span_ratio']].sort_values('noise_span_ratio',ascending=False).head(10))
fig,axes=plt.subplots(2,3,figsize=(16,8))
for ax,c in zip(axes.flat,FLEX):
    d=cal[cal.finger==c]; x=np.arange(len(d)); ax.plot(x,d.neutral_mean,'o-',label='neutral mean'); ax.plot(x,d.fist_median,'o-',label='fist median')
    ax.set_xticks(x,d.participant+'/'+d.session,rotation=90); ax.set_title(c); ax.set_ylabel('ADC'); ax.legend()
axes.flat[-1].axis('off'); plot_save('neutral_vs_fist_by_session')
# Report unusual reference directions and small empirical spans without imposing a universal minimum.
cal_review=[]
for finger,d in cal.groupby('finger'):
    majority=float(np.sign(d.signed_span).mode().iloc[0])
    q1,q3=d.absolute_span.quantile([.25,.75]); lower=q1-1.5*(q3-q1)
    for _,r in d.iterrows():
        reasons=[]
        if np.sign(r.signed_span)!=majority: reasons.append('Direction differs from majority of session references for this finger')
        if r.absolute_span<lower: reasons.append('Span below Q1 - 1.5 IQR (exploratory session-level fence)')
        if reasons: cal_review.append({'participant':r.participant,'session':r.session,'finger':finger,'signed_span':r.signed_span,'reason':'; '.join(reasons)})
cal_review=pd.DataFrame(cal_review,columns=['participant','session','finger','signed_span','reason'])
save(cal_review,'calibration_review.csv','Reference direction and small-span warnings; a difference is not proof of sensor polarity reversal.')
display(cal_review)
# Reference availability must precede accepted data for prospective calibration use.
cal_times=calmeta[RKEY+['time']].rename(columns={'time':'calibration_time'})
reference_timing=index.merge(cal_times,on=RKEY,how='left',validate='many_to_one')
reference_timing['calibration_before_trial']=pd.to_datetime(reference_timing.calibration_time)<=pd.to_datetime(reference_timing.time)
save(reference_timing[KEY+['session','ref_id','time','calibration_time','calibration_before_trial']],
     'calibration_timing.csv','Whether each accepted trial follows its mapped calibration; fist event timing is listed separately below.')
fist_timing=events[events.event=='FIST_REFERENCE'][['participant','session','time']].rename(columns={'time':'fist_time'})
if fist_timing.duplicated(['participant','session']).any(): raise RuntimeError('Ambiguous fist event time')
ft=index.merge(fist_timing,on=['participant','session'],how='left',validate='many_to_one')
ft['fist_before_trial']=pd.to_datetime(ft.fist_time)<=pd.to_datetime(ft.time)
save(ft[KEY+['session','time','fist_time','fist_before_trial']],'fist_reference_timing.csv','Prospective availability of the session fist reference.')
print('Calibrations before accepted trials:',int(reference_timing.calibration_before_trial.sum()),'/',len(index))
print('Fist references before accepted trials:',int(ft.fist_before_trial.sum()),'/',len(index))
calshift=cal.sort_values(['participant','finger','session']).copy()
calshift['neutral_change_from_previous_session']=calshift.groupby(['participant','finger']).neutral_mean.diff()
save(calshift,'calibration_session_changes.csv','Baseline changes across recalibration sessions; not assumed to be electronic drift.')


## Phase 7 — Raw IMU units and stationary-hold quality

**A. Purpose.** Preserve raw values and compute physically interpretable motion summaries.

**C. Expected output.** Per-trial gyro median/peak/IQR and acceleration magnitude median/std/range.

**D. Interpretation.** Acceleration includes gravity; norms near 1 g are consistent with a static sensor but are not proof of zero motion.

**E. Genuine problem.** Values outside the signed 16-bit raw encoding or nonfinite conversions require an integrity review.

**F. Warning only.** High gyro readings may reflect motion and bias; robust distributional flags prompt trace review.

**G. Do not conclude.** Uncorrected gyro magnitude is not pure motion: calibrated gyro biases are reported separately.

**B. Runnable code:**


In [ ]:
physical=units(raw)
imu=[]
for key,d in physical.groupby(KEY,sort=True):
    row=dict(zip(KEY,key))
    for name,c in [('gyro','gyro_magnitude_dps'),('accel','accel_magnitude_g')]:
        row[name+'_mag_median']=d[c].median(); row[name+'_mag_max']=d[c].max()
        row[name+'_mag_iqr']=d[c].quantile(.75)-d[c].quantile(.25)
        row[name+'_mag_std']=d[c].std(); row[name+'_mag_range']=d[c].max()-d[c].min()
    for c in ['ax_g','ay_g','az_g']: row[c+'_median']=d[c].median()
    imu.append(row)
T=T.merge(pd.DataFrame(imu),on=KEY,validate='one_to_one')
imu_oob=((raw[['gx','gy','gz','ax','ay','az']] < -32768)|(raw[['gx','gy','gz','ax','ay','az']]>32767)).any(axis=1)
check('raw IMU signed 16-bit bounds',0,int(imu_oob.sum()),not imu_oob.any()); flush_audit()
display(T[['gyro_mag_median','gyro_mag_max','gyro_mag_iqr','accel_mag_median','accel_mag_std','accel_mag_range']].describe(percentiles=[.5,.95,.99]).T)
fig,axes=plt.subplots(1,3,figsize=(13,4))
for ax,c in zip(axes,['gyro_mag_max','accel_mag_median','accel_mag_std']): ax.hist(T[c],bins=30); ax.set_xlabel(c); ax.set_ylabel('Trials')
plot_save('imu_trial_distributions')


## Phase 8 — Quaternion validity

**A. Purpose.** Inspect raw norms before normalization.

**C. Expected output.** Norm range and mean, p95, maximum error; invalid zero quaternions explicitly stop orientation analysis.

**D. Interpretation.** Normalization makes angular calculations valid numerically; original norm error remains a separate QC variable.

**E. Genuine problem.** Zero or nonfinite quaternion cannot represent an orientation.

**F. Warning only.** A small empirical norm-error outlier is not automatically corrupt; no universal norm tolerance is invented.

**G. Do not conclude.** Unit length does not prove orientation accuracy or absence of drift.

**B. Runnable code:**


In [ ]:
qnorm=np.linalg.norm(raw[Q],axis=1); qerr=np.abs(qnorm-1)
q_summary=pd.DataFrame([{'norm_min':qnorm.min(),'norm_max':qnorm.max(),'error_mean':qerr.mean(),
                        'error_p95':np.percentile(qerr,95),'error_max':qerr.max(),'zero_norm_rows':int((qnorm==0).sum())}])
save(q_summary,'quaternion_norm_summary.csv','Raw quaternion validity before any normalization.'); display(q_summary)
check('nonzero quaternion norm',0,int((qnorm==0).sum()),not (qnorm==0).any()); flush_audit()
if (qnorm==0).any(): raise RuntimeError('Zero quaternion: orientation calculations stopped.')
plt.figure(figsize=(7,4)); plt.hist(qerr,bins=50); plt.xlabel('|norm(q) - 1|'); plt.ylabel('Samples (descriptive)'); plot_save('quaternion_norm_error')


## Phase 9 — Within-trial quaternion motion

**A. Purpose.** Use normalized sign-invariant quaternion distances to distinguish steady holds from motion.

**C. Expected output.** P95 and maximum angular distance from Markley mean, pairwise diameter, and end-to-start change.

**D. Interpretation.** The p95 distance reduces sensitivity to single spikes; the Markley center itself is a valid rotation mean, not claimed to be a robust estimator.

**E. Genuine problem.** No automatic rejection occurs without a confirmed protocol violation or trace evidence.

**F. Warning only.** Large diameter, flex range and gyro peaks are reviewed jointly; different logged spread definitions may give different values.

**G. Do not conclude.** Small 2-second motion does not establish stability over a 20-minute session.

**B. Runnable code:**


In [ ]:
orientation=pd.DataFrame([dict(zip(KEY,key),**q_metrics(d)) for key,d in raw.groupby(KEY,sort=True)])
T=T.merge(orientation,on=KEY,validate='one_to_one')
T['flex_range_max_recomputed']=T[[c+'_range' for c in FLEX]].max(axis=1)
T['flex_range_log_difference']=T.flex_range_max_recomputed-T.flex_range_max
T['gyro_peak_log_difference']=T.gyro_mag_max-T.max_dps
T['orientation_p95_minus_logged_spread']=T.orientation_p95_deg-T.spread_deg
# Bias-corrected comparison is separate: firmware may calculate peaks on a different sample stream.
biascheck=[]
for key,d in physical.groupby(KEY,sort=True):
    rr=T.loc[(T.participant==key[0])&(T.gesture_id==key[1])&(T.trial_id==key[2])].iloc[0]
    b=calmeta[(calmeta.participant==rr.participant)&(calmeta.session==rr.session)&(calmeta.ref_id==rr.ref_id)].iloc[0]
    bias=b[['gyro_bias_x_dps','gyro_bias_y_dps','gyro_bias_z_dps']].to_numpy(float)
    corrected_peak=np.linalg.norm(d[['gx_dps','gy_dps','gz_dps']].to_numpy()-bias,axis=1).max()
    biascheck.append(dict(zip(KEY,key),gyro_bias_corrected_peak=corrected_peak))
T=T.merge(pd.DataFrame(biascheck),on=KEY,validate='one_to_one')
T['gyro_corrected_peak_log_difference']=T.gyro_bias_corrected_peak-T.max_dps
save(T[KEY+['session','max_dps','gyro_mag_max','gyro_bias_corrected_peak','gyro_peak_log_difference','gyro_corrected_peak_log_difference']],
     'gyro_peak_definition_review.csv','Raw and bias-corrected sampled peaks versus firmware logs; mismatches do not prove incorrect units without firmware timing/definition evidence.')
print('Largest raw peak/log disagreement:',T.gyro_peak_log_difference.abs().max(),'deg/s; inspect firmware peak calculation.')

save(T[KEY+['session','orientation_p95_deg','orientation_diameter_deg','gyro_mag_max','flex_range_max_recomputed','spread_deg','max_dps','flex_range_log_difference','gyro_peak_log_difference']],
     'motion_log_comparison.csv','Recomputed motion and logged QC; angular definitions and gyro bias handling may differ.')
display(T[['orientation_p95_deg','orientation_diameter_deg','orientation_end_to_start_deg','gyro_peak_log_difference','flex_range_log_difference']].describe(percentiles=[.5,.95,.99]).T)
fig,ax=plt.subplots(figsize=(7,5)); im=ax.scatter(T.gyro_mag_max,T.orientation_diameter_deg,c=T.flex_range_max_recomputed,cmap='viridis'); ax.set_xlabel('Raw gyro peak (deg/s)'); ax.set_ylabel('Quaternion pairwise diameter (deg)'); fig.colorbar(im,ax=ax,label='Largest finger range (ADC)'); plot_save('motion_crosscheck')


## Phase 10 — Checkpoint orientation deviation and final drift

**A. Purpose.** Compare every checkpoint and final drift file only with its own session reference.

**C. Expected output.** Per-file quaternion deviation, logged drift comparison and session-separated plots.

**D. Interpretation.** Neutral return deviation mixes orientation estimator drift and physical failure to reproduce the neutral pose. Without ground truth, report it as observed deviation.

**E. Genuine problem.** Ambiguous reference selection stops calculation. A final check in another session cannot certify the earlier chain.

**F. Warning only.** Large deviation threatens raw absolute orientation features, even if each trial is steady.

**G. Do not conclude.** Do not equate the full rotation angle with yaw drift; no external orientation truth or firmware convention is supplied.

**B. Runnable code:**


In [ ]:
drift_rows=[]
for p in P_DIRS:
    files=sorted(list(p.glob('checkpoint_*.csv'))+list(p.glob('drift_check_*.csv')))
    for f in files:
        session=re.search(r'S\d+',f.name)[0]; d=cache[f]
        ref=refs[(refs.participant==p.name)&(refs.session==session)]
        if len(ref)!=1: raise RuntimeError(f'{p.name}/{session}: ambiguous checkpoint reference; need event/ref identity')
        e=events[(events.participant==p.name)&(events.session==session)&events.details.fillna('').str.contains('file='+f.name,regex=False)]
        if len(e)>1: raise RuntimeError('Multiple event matches for checkpoint '+str(f))
        details=e.iloc[0].details if len(e) else ''
        m=re.search(r'drift_deg=([-+\d.]+)',details)
        logged=float(m[1]) if m else np.nan
        observed=float(q_angle(q_mean(d[Q]),ref[Q].to_numpy()[0]))
        rnd=re.search(r'_R(\d+)',f.name)
        drift_rows.append(dict(participant=p.name,session=session,ref_id=ref.iloc[0].ref_id,file=f.name,
            kind='checkpoint' if rnd else 'final',round=int(rnd[1]) if rnd else np.nan,
            time=e.iloc[0].time if len(e) else None,drift_deg=observed,logged_drift_deg=logged,
            logged_minus_recomputed=logged-observed,within_hold_p95_deg=q_metrics(d)['orientation_p95_deg'],
            accepted_trials_in_session=int(((index.participant==p.name)&(index.session==session)).sum()),
            event_matches=len(e)))
drift=pd.DataFrame(drift_rows)
save(drift,'checkpoint_drift.csv','Same-session angular deviation for checkpoints and final holds; includes log cross-check and session relevance.')
display(drift.groupby(['participant','session']).agg(n_files=('file','size'),min_deg=('drift_deg','min'),max_deg=('drift_deg','max'),max_log_disagreement=('logged_minus_recomputed',lambda x:x.abs().max())).reset_index())
fig,axes=plt.subplots(2,5,figsize=(17,7),sharey=True)
for ax,pid in zip(axes.flat,PARTS):
    for session,d in drift[(drift.participant==pid)&(drift.kind=='checkpoint')].groupby('session'):
        d=d.sort_values('round'); ax.plot(d['round'],d.drift_deg,'o-',label=session)
    ax.set_title(pid); ax.set_xlabel('Checkpoint round'); ax.set_ylabel('Neutral return angle (deg)'); ax.legend(fontsize=8)
plot_save('checkpoint_drift_by_session')
finaldrift=drift[drift.kind=='final'].copy()
fig,ax=plt.subplots(figsize=(10,4)); ax.bar(finaldrift.participant+'/'+finaldrift.session,finaldrift.drift_deg); ax.tick_params(axis='x',rotation=45); ax.set_ylabel('Final return angle (deg)'); ax.set_title('Session-specific final check; inspect accepted_trials_in_session'); plot_save('final_drift')
display(finaldrift[['participant','session','drift_deg','accepted_trials_in_session']])


## Phase 11 — Flex baseline change at neutral checkpoints

**A. Purpose.** Quantify neutral flex displacement independently of quaternion deviation.

**C. Expected output.** Checkpoint medians minus the matching session’s neutral mean, per finger.

**D. Interpretation.** Persistent shifts can reflect glove fit, hand posture or electrical baseline change.

**E. Genuine problem.** An unmatched session baseline makes its delta undefined and must not be filled from S01.

**F. Warning only.** Nonzero deltas alone do not establish sensor failure.

**G. Do not conclude.** Do not identify a unique physical cause from baseline shifts alone.

**B. Runnable code:**


In [ ]:
flexdrift=[]
for r in drift.itertuples():
    d=cache[ROOT/'raw'/r.participant/r.file]
    for c in FLEX:
        baseline=cal[(cal.participant==r.participant)&(cal.session==r.session)&(cal.finger==c)]
        if len(baseline)!=1: raise RuntimeError('Missing/ambiguous flex baseline')
        base=baseline.iloc[0]
        flexdrift.append(dict(participant=r.participant,session=r.session,kind=r.kind,round=r.round,file=r.file,
          finger=c,checkpoint_median=d[c].median(),neutral_mean=base.neutral_mean,
          delta_adc=d[c].median()-base.neutral_mean,delta_fraction_span=(d[c].median()-base.neutral_mean)/base.signed_span if base.signed_span else np.nan))
flexdrift=pd.DataFrame(flexdrift)
save(flexdrift,'checkpoint_flex_drift.csv','Neutral flex change in ADC and signed fist-span units; final holds kept distinct.')
heatmap(flexdrift.pivot_table(index=['participant','session'],columns='finger',values='delta_adc',aggfunc=lambda s:s.abs().max()),'Largest absolute neutral baseline change (ADC)','checkpoint_flex_max_change')
fig,axes=plt.subplots(2,5,figsize=(17,7))
for ax,pid in zip(axes.flat,PARTS):
    for (session,finger),d in flexdrift[(flexdrift.participant==pid)&(flexdrift.kind=='checkpoint')].groupby(['session','finger']):
        d=d.sort_values('round'); ax.plot(d['round'],d.delta_adc,marker='.',label=session+'/'+finger.replace('flex_',''))
    ax.set_title(pid); ax.set_xlabel('Round'); ax.set_ylabel('Delta ADC')
fig.legend(*axes.flat[1].get_legend_handles_labels(),loc='outside right upper',fontsize=7)
plot_save('checkpoint_flex_rounds')


## Phase 12 — Round, temperature and collection sequence

**A. Purpose.** Explore temporal association while accounting descriptively for participant, gesture and session offsets.

**C. Expected output.** Within-participant/session, gesture-demeaned correlations; round trajectories; actual position balance.

**D. Interpretation.** Correlations are computed separately for each participant/session. Demeaning removes cell offsets but cannot fully disentangle time, temperature and fatigue.

**E. Genuine problem.** Wrong planned-versus-observed order would already appear in the provenance audit.

**F. Warning only.** Few trials or constant temperature make a correlation undefined. No p-values are used to pretend trials from one person are independent subjects.

**G. Do not conclude.** Correlation cannot prove thermal causation; planned balance must be verified rather than assumed.

**B. Runnable code:**


In [ ]:
T['collection_time']=pd.to_datetime(T.time,errors='coerce')
T=T.sort_values(['participant','collection_time','log_row']).reset_index(drop=True)
T['collection_sequence']=T.groupby('participant').cumcount()+1
T['elapsed_session_minutes']=(T.collection_time-T.groupby(['participant','session']).collection_time.transform('min')).dt.total_seconds()/60
T['elapsed_participant_minutes']=(T.collection_time-T.groupby('participant').collection_time.transform('min')).dt.total_seconds()/60
position_balance=pd.crosstab(index.gesture_id,index.position)
save(position_balance.reset_index(),'gesture_position_balance.csv','Observed accepted gesture × position counts; verifies rather than assumes balance.')
heatmap(position_balance,'Observed gesture × order position','order_balance',True)
assoc=[]
for (pid,session),d in T.groupby(['participant','session']):
    for feat in [c+'_median' for c in FLEX]+['gyro_mag_max','orientation_p95_deg']:
        for cov in ['round','collection_sequence','elapsed_session_minutes','temp_c']:
            xx=d[cov]-d.groupby('gesture_id')[cov].transform('mean')
            yy=d[feat]-d.groupby('gesture_id')[feat].transform('mean')
            ok=np.isfinite(xx)&np.isfinite(yy)
            rho=float(spearmanr(xx[ok],yy[ok]).statistic) if ok.sum()>=3 and xx[ok].nunique()>1 and yy[ok].nunique()>1 else np.nan
            assoc.append(dict(participant=pid,session=session,feature=feat,covariate=cov,n_trials=len(d),rho=rho))
assoc=pd.DataFrame(assoc)
save(assoc,'temporal_associations.csv','Descriptive Spearman correlations after within-participant/session/gesture demeaning; no causality or independent-trial inference.')
display(assoc.groupby(['feature','covariate']).rho.agg(['count','median','min','max']).reset_index())
fig,axes=plt.subplots(1,2,figsize=(13,4))
for (pid,session),d in T.groupby(['participant','session']):
    d=d.sort_values('collection_sequence'); axes[0].plot(d.collection_sequence,d.temp_c,'.-',label=pid+'/'+session)
    residual=d.flex_thumb_median-d.groupby('gesture_id').flex_thumb_median.transform('mean')
    axes[1].plot(d['round'],residual,'.',alpha=.6)
axes[0].set(xlabel='Accepted sequence within participant',ylabel='Temperature (°C)')
axes[1].set(xlabel='Round',ylabel='Thumb residual within session/gesture (ADC)')
plot_save('temporal_effects')


## Phase 13 — One row per accepted trial

**A. Purpose.** Assemble the complete raw feature table and explicit feature lists.

**C. Expected output.** Exactly 500 unique composite keys, original logged metrics and derived variables.

**D. Interpretation.** Identity and log fields are available for QC but are not accidentally passed to PCA.

**E. Genuine problem.** Duplicate keys or nonfinite analysis features stop descriptive embeddings.

**F. Warning only.** Not every field in this table is an appropriate future predictor.

**G. Do not conclude.** Trial IDs, participant IDs, time, temperature, order, session and reference IDs should not become accidental classifier shortcuts.

**B. Runnable code:**


In [ ]:
FLEX_MED=[c+'_median' for c in FLEX]
EDA_FEATURES=FLEX_MED+[c+'_iqr' for c in FLEX]+['ax_g_median','ay_g_median','az_g_median',
    'accel_mag_median','accel_mag_std','gyro_mag_median','gyro_mag_max','orientation_p95_deg']
assert len(T)==500 and not T.duplicated(KEY).any()
assert np.isfinite(T[EDA_FEATURES]).all().all()
save(T,'trial_level_features_raw.csv','500 trial-level records: identity, provenance, all raw summaries and descriptive QC features; no classifier trained.')
display(T[KEY+['session']+FLEX_MED+['gyro_mag_max','orientation_p95_deg']].head(10))
print('Trial rows:',len(T),'| participants:',T.participant.nunique(),'| columns:',len(T.columns))


## Phase 14 — Participant versus gesture variation

**A. Purpose.** Compare the same gesture across people and quantify participant offsets on a common scale.

**C. Expected output.** Five participant × gesture heatmaps, gesture profiles, and a balanced two-factor descriptive sum-of-squares decomposition.

**D. Interpretation.** For each finger, separate participant main effect, gesture main effect, participant×gesture interaction and within-cell repetition variation.

**E. Genuine problem.** Large subject effects indicate difficulty for uncalibrated cross-person features, not data corruption.

**F. Warning only.** Interaction means the gesture pattern itself differs across people, which simple baseline subtraction may not fix.

**G. Do not conclude.** These are descriptive variance fractions, not causal effects, significance tests, or unseen-person accuracy.

**B. Runnable code:**


In [ ]:
def variation_table(df, features):
    records=[]
    for c in features:
        d=df[['participant','gesture_id',c]].dropna(); grand=d[c].mean()
        pp=d.groupby('participant')[c].mean(); gg=d.groupby('gesture_id')[c].mean()
        cc=d.groupby(['participant','gesture_id'])[c].agg(['mean','size'])
        ss_total=((d[c]-grand)**2).sum()
        ss_p=sum(d.participant.eq(p).sum()*(v-grand)**2 for p,v in pp.items())
        ss_g=sum(d.gesture_id.eq(g).sum()*(v-grand)**2 for g,v in gg.items())
        ss_int=sum(r['size']*(r['mean']-pp.loc[p]-gg.loc[g]+grand)**2 for (p,g),r in cc.iterrows())
        pred=d.set_index(['participant','gesture_id']).index.map(cc['mean'])
        ss_within=np.sum((d[c].to_numpy()-np.asarray(pred))**2)
        records.append(dict(feature=c,participant_fraction=ss_p/ss_total,gesture_fraction=ss_g/ss_total,
             interaction_fraction=ss_int/ss_total,within_cell_fraction=ss_within/ss_total,
             decomposition_error=(ss_p+ss_g+ss_int+ss_within-ss_total)/ss_total))
    return pd.DataFrame(records)
variation=variation_table(T,FLEX_MED)
save(variation,'variation_decomposition_raw.csv','Balanced descriptive sums of squares; no independent-sample F tests.'); display(variation)
fig,axes=plt.subplots(2,3,figsize=(16,8))
for ax,c in zip(axes.flat,FLEX_MED):
    table=T.pivot_table(index='participant',columns='gesture_id',values=c,aggfunc='median').reindex(index=PARTS,columns=GEST)
    im=ax.imshow(table,aspect='auto'); ax.set_xticks(range(10),GEST,rotation=45); ax.set_yticks(range(10),PARTS); ax.set_title(c); fig.colorbar(im,ax=ax,label='ADC')
axes.flat[-1].axis('off'); plot_save('participant_gesture_flex_heatmaps')
profiles=T.groupby('gesture_id')[FLEX_MED].median()
profiles.T.plot(figsize=(11,5),marker='o'); plt.ylabel('Trial median ADC'); plt.xlabel('Finger'); plot_save('gesture_flex_profiles')


## Phase 15 — Within-class dispersion and between-class differences

**A. Purpose.** Identify gesture pairs with small centroid distance relative to their within-class spread.

**C. Expected output.** Nearest gesture pairs and within-gesture RMS radii, in globally standardized five-flex-median space.

**D. Interpretation.** Distance divided by the sum of the two RMS radii is a descriptive separation ratio; lower values deserve a closer look.

**E. Genuine problem.** No numerical ratio is treated as a pass/fail criterion.

**F. Warning only.** Participant variation and multimodal classes can hide within-class structure.

**G. Do not conclude.** Centroid distance and apparent overlap are not classification accuracy or proof of wrong labels.

**B. Runnable code:**


In [ ]:
Xflex=StandardScaler().fit_transform(T[FLEX_MED])
centroids={g:Xflex[T.gesture_id.eq(g)].mean(axis=0) for g in GEST}
radii={g:np.sqrt(np.mean(np.sum((Xflex[T.gesture_id.eq(g)]-centroids[g])**2,axis=1))) for g in GEST}
pairs=[]
for i,g in enumerate(GEST):
    for h in GEST[i+1:]:
        dist=np.linalg.norm(centroids[g]-centroids[h]); denom=radii[g]+radii[h]
        pairs.append(dict(gesture_a=g,name_a=NAME_MAP[g],gesture_b=h,name_b=NAME_MAP[h],centroid_distance=dist,
                          rms_radius_a=radii[g],rms_radius_b=radii[h],separation_ratio=dist/denom if denom else np.nan))
pairs=pd.DataFrame(pairs).sort_values('separation_ratio')
save(pairs,'gesture_separation_exploratory.csv','All pairwise centroid-to-dispersion ratios in standardized raw flex space; no accuracy claims.')
display(pairs.head(10))
heatmap(pd.DataFrame(cdist(np.array(list(centroids.values())),np.array(list(centroids.values()))),index=GEST,columns=GEST),
        'Gesture centroid distances: standardized raw flex','gesture_centroid_distances')


## Phase 16 — Correlation and redundant variables

**A. Purpose.** Inspect associations among trial-level flex, acceleration, gyro and motion features.

**C. Expected output.** Spearman matrix, strong pairs and a within-participant demeaned matrix for comparison.

**D. Interpretation.** Large pooled correlations can arise from subject offsets or gesture structure.

**E. Genuine problem.** Constant variables need separate treatment because their correlation is undefined.

**F. Warning only.** Absolute correlation ≥0.90 is a descriptive reporting heuristic only.

**G. Do not conclude.** Do not remove features during EDA or treat correlated channels as interchangeable in all gestures.

**B. Runnable code:**


In [ ]:
corr=T[EDA_FEATURES].corr(method='spearman')
heatmap(corr,'Trial-level Spearman correlations','feature_correlations')
strong=[]
for i,a in enumerate(EDA_FEATURES):
    for b in EDA_FEATURES[i+1:]:
        if abs(corr.loc[a,b])>=.90: strong.append({'feature_a':a,'feature_b':b,'rho':corr.loc[a,b]})
save(pd.DataFrame(strong,columns=['feature_a','feature_b','rho']),'strong_correlations.csv','Candidate redundancy using exploratory |Spearman rho| >= .90.'); display(pd.DataFrame(strong))
centered=T[EDA_FEATURES]-T.groupby('participant')[EDA_FEATURES].transform('mean')
save(centered.corr(method='spearman').reset_index(),'within_participant_correlations.csv','Demeaned descriptive correlation matrix to compare with pooled correlations.')


## Phase 17 — Descriptive PCA and subject structure

**A. Purpose.** Visualize the same standardized trial features colored separately by gesture and participant.

**C. Expected output.** Explained variance and identical PC1/PC2 coordinates in two plots; silhouette comparisons in full standardized space.

**D. Interpretation.** PCA on the full dataset is for EDA only. Full-space silhouette and two-axis scatter show different aspects of separation.

**E. Genuine problem.** Nonfinite input or accidental identity predictors invalidate the embedding.

**F. Warning only.** PCA may miss nonlinear or low-variance discriminative information. Scaling choices affect distances.

**G. Do not conclude.** No classification accuracy, no validation score and no optimal feature set can be inferred from these plots.

**B. Runnable code:**


In [ ]:
X=StandardScaler().fit_transform(T[EDA_FEATURES]); pca=PCA().fit(X); coords=pca.transform(X)
pcs=T[KEY+['session']].copy(); pcs['PC1']=coords[:,0]; pcs['PC2']=coords[:,1]
save(pcs,'pca_coordinates.csv','Descriptive full-dataset standardized PCA; refit preprocessing inside future training folds.')
save(pd.DataFrame({'component':np.arange(1,len(pca.explained_variance_ratio_)+1),'explained_variance_ratio':pca.explained_variance_ratio_}),
     'pca_variance.csv','Variance explained, not class prediction performance.')
fig,axes=plt.subplots(1,3,figsize=(17,5))
axes[0].plot(np.arange(1,len(pca.explained_variance_ratio_)+1),np.cumsum(pca.explained_variance_ratio_),'o-'); axes[0].set(xlabel='Number of PCs',ylabel='Cumulative explained variance',ylim=(0,1.02))
for ax,group in zip(axes[1:],['gesture_id','participant']):
    for i,label in enumerate(sorted(T[group].unique())):
        mask=T[group].eq(label).to_numpy(); ax.scatter(coords[mask,0],coords[mask,1],s=16,alpha=.65,color=plt.cm.tab10(i),label=label)
    ax.set(xlabel=f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',ylabel=f'PC2 ({pca.explained_variance_ratio_[1]:.1%})',title='Color = '+group); ax.legend(fontsize=7,ncol=2)
plot_save('pca_gesture_participant')
sil=pd.DataFrame([{'representation':'all EDA features','labels':label,'silhouette':silhouette_score(X,T[label])} for label in ['gesture_id','participant']])
save(sil,'silhouette_exploratory.csv','Exploratory full-standardized-space label compactness; never accuracy.'); display(sil)
if sil.set_index('labels').loc['participant','silhouette']>sil.set_index('labels').loc['gesture_id','silhouette']:
    print('REVIEW: participant-label compactness exceeds gesture-label compactness in this feature space; possible subject effect.')
else: print('Gesture silhouette is at least as high as participant silhouette here; this does not establish subject independence.')


## Phase 18 — Separate exploratory calibration representations

**A. Purpose.** Compare raw, neutral-subtracted and signed-span-scaled flex without overwriting raw features.

**C. Expected output.** Span/noise distribution, denominator eligibility and comparable participant/gesture variance fractions and silhouettes.

**D. Interpretation.** Eligibility uses |span| > 3 × max(neutral SD, fist SD), with a one-ADC-count resolution floor. This is an explicitly exploratory noise heuristic, not a validated cutoff; sensitivity at 1×, 3× and 5× is shown.

**E. Genuine problem.** Zero denominator is mathematically invalid. Missing or too-small spans remain NaN; no invented substitution.

**F. Warning only.** Calibration from a new participant may be permissible only if deployment will also collect it. Calibration after a trial is retrospective, not proof of deployable normalization.

**G. Do not conclude.** Improved appearance does not prove better unseen-participant classification. Compare only common eligible trials.

**B. Runnable code:**


In [ ]:
sensitivity=[]
for k in [1,3,5]:
    eligible=cal.absolute_span.gt(np.maximum(1,k*cal[['neutral_std','fist_std']].max(axis=1)))
    sensitivity.append({'noise_multiplier_heuristic':k,'eligible_session_fingers':int(eligible.sum()),'total':len(cal)})
display(pd.DataFrame(sensitivity)); display(cal[['absolute_span','noise_span_ratio']].describe(percentiles=[.05,.5,.95]))
explore=T[KEY+['session','ref_id']+FLEX_MED].copy()
for c in FLEX:
    base=cal[cal.finger==c][RKEY+['neutral_mean','signed_span','neutral_std','fist_std']]
    m=T[RKEY].merge(base,on=RKEY,how='left',validate='many_to_one')
    floor=np.maximum(1,3*m[['neutral_std','fist_std']].max(axis=1))
    valid=m.signed_span.abs().gt(floor) & m.signed_span.notna()
    explore[c+'_neutral_subtracted']=T[c+'_median'].to_numpy()-m.neutral_mean.to_numpy()
    explore[c+'_span_scaled']=np.where(valid,explore[c+'_neutral_subtracted']/m.signed_span,np.nan)
    explore[c+'_span_eligible']=valid.to_numpy()
save(explore,'trial_features_exploratory_normalized.csv','Separate retrospective calibration comparisons; signed spans preserved, invalid denominators left NaN.')
common=explore[[c+'_span_scaled' for c in FLEX]].notna().all(axis=1)
comparison=[]; norm_sil=[]
for name,cols in [('raw',FLEX_MED),('neutral_subtracted',[c+'_neutral_subtracted' for c in FLEX]),('span_scaled',[c+'_span_scaled' for c in FLEX])]:
    d=explore.loc[common,['participant','gesture_id']+cols].copy()
    # Sum-of-squares decomposition requires the same complete balanced design.
    counts=pd.crosstab(d.participant,d.gesture_id)
    if len(d)==500 and counts.eq(5).all().all():
        v=variation_table(d,cols); v['representation']=name; comparison.append(v)
    if len(d)>10:
        xx=StandardScaler().fit_transform(d[cols])
        for label in ['gesture_id','participant']:
            norm_sil.append(dict(representation=name,labels=label,n_trials=len(d),silhouette=silhouette_score(xx,d[label])))
comparison=pd.concat(comparison,ignore_index=True) if comparison else pd.DataFrame()
norm_sil=pd.DataFrame(norm_sil)
save(comparison,'normalization_variation_comparison.csv','Same-cohort descriptive variance fractions; omitted if eligibility destroys complete balance.')
save(norm_sil,'normalization_silhouette_comparison.csv','Same-cohort full-space silhouette; no claim of predictive improvement.')
display(comparison); display(norm_sil)
if not comparison.empty:
    comparison.groupby('representation')[['participant_fraction','gesture_fraction','interaction_fraction','within_cell_fraction']].mean().plot(kind='bar',stacked=True,figsize=(10,5)); plt.ylabel('Mean descriptive variance fraction across fingers'); plot_save('normalization_comparison')


## Phase 19 — Traceable QC flags, with no automatic deletion

**A. Purpose.** Create an auditable review list that distinguishes objective errors from statistical extremes.

**C. Expected output.** One suspicious-trial row with JSON arrays for all flags, plus a long table with one row per flag and thresholds.

**D. Interpretation.** Upper Tukey fences use Q3 + 3 IQR globally and within gesture; acceleration uses both tails. These conservative descriptive fences are heuristic. Zero-IQR distributions use no empirical fence.

**E. Genuine problem.** ADC/IMU encoding violations and zero quaternion are hard errors. Log identity problems stop earlier sections.

**F. Warning only.** Exact rails, constant channels, unusual motion, step changes, timing delays and norm errors are review flags, not automatic exclusions. Multiple metrics create multiple correlated flags.

**G. Do not conclude.** Flag counts do not estimate the number of bad trials. Unflagged does not mean correctly labeled.

**B. Runnable code:**


In [ ]:
flags=[]; thresholds=[]
def flag_row(row,metric,value,threshold,reason,ftype='STATISTICAL_OUTLIER',severity='REVIEW'):
    flags.append({**{k:row[k] for k in KEY+['session']},'metric':metric,'observed_value':float(value),
      'reference_distribution_or_threshold':threshold,'flag_reason':reason,'flag_type':ftype,'severity':severity})
metrics=[c+'_range' for c in FLEX]+[c+'_max_step' for c in FLEX]+['gyro_mag_max','orientation_p95_deg','orientation_diameter_deg','q_norm_error_max','accel_mag_median','accel_mag_std','late_max_us']
for metric in metrics:
    groups=[('global',T)]+[('gesture '+g,d) for g,d in T.groupby('gesture_id')]
    for label,d in groups:
        lo,hi=d[metric].quantile([.25,.75]); iq=hi-lo
        lower=lo-3*iq if metric=='accel_mag_median' else -np.inf
        upper=hi+3*iq
        thresholds.append(dict(metric=metric,distribution=label,n_trials=len(d),q1=lo,q3=hi,iqr=iq,lower=lower,upper=upper,applied=bool(iq>0)))
        if iq<=0: continue
        for _,row in d[(d[metric]>upper)|(d[metric]<lower)].iterrows():
            flag_row(row,metric,row[metric],f'{label}: Q1-3IQR={lower:.8g}; Q3+3IQR={upper:.8g}',
                     'Exploratory distributional extreme; inspect trace and context, do not delete.')
for _,row in T.iterrows():
    for c in FLEX:
        if row[c+'_outside_samples']>0: flag_row(row,c+'_outside_samples',row[c+'_outside_samples'],'0..4095 stated encoding','Out-of-range ADC value','HARD_INTEGRITY_ERROR','FAIL')
        if row[c+'_rail_samples']>0: flag_row(row,c+'_rail_samples',row[c+'_rail_samples'],'exactly 0 or 4095','ADC rail occupancy; possible clipping','POSSIBLE_SENSOR_ISSUE')
        if row[c+'_unique_values']==1: flag_row(row,c+'_unique_values',1,'constant for all 50 samples','Possible stuck/quantized static channel; compare other trials','POSSIBLE_SENSOR_ISSUE')
    if row.interval_min_ms!=40 or row.interval_max_ms!=40 or row.imu_missed!=0:
        flag_row(row,'imu_missed',row.imu_missed,'0 missed; logged intervals 40 ms','Logged timing exception','PROTOCOL_WARNING')
    if row.flex_range_log_difference!=0:
        flag_row(row,'flex_range_log_difference',row.flex_range_log_difference,'exact integer range agreement','Recomputed flex range differs from logged value','LOG_INCONSISTENCY')
    logged_sat=str(row.flex_saturated).lower() in ['true','1','1.0']
    actual_sat=sum(row[c+'_rail_samples'] for c in FLEX)>0
    if logged_sat!=actual_sat: flag_row(row,'flex_saturated',int(logged_sat),'compare logged saturation and exact rail occupancy','Definitions may differ: inspect firmware saturation criterion','LOG_INCONSISTENCY')
flagcols=KEY+['session','metric','observed_value','reference_distribution_or_threshold','flag_reason','flag_type','severity']
flaglong=pd.DataFrame(flags,columns=flagcols)
save(flaglong,'qc_flags_long.csv','One row per triggered metric and comparison distribution, allowing multiple flags per trial.')
compact=[]
for key,d in flaglong.groupby(KEY+['session'],sort=True):
    row=dict(zip(KEY+['session'],key))
    for c in ['metric','observed_value','reference_distribution_or_threshold','flag_reason','flag_type','severity']:
        row[c]=json.dumps(d[c].tolist())
    row['n_flags']=len(d); compact.append(row)
flagged=pd.DataFrame(compact,columns=flagcols+['n_flags'])
save(flagged,'qc_flagged_trials.csv','One row per suspicious accepted trial; aligned JSON arrays preserve every metric/value/threshold/reason/type/severity.')
save(pd.DataFrame(thresholds),'qc_thresholds.csv','All empirical 3-IQR fences and group sizes; exploratory rather than acquisition acceptance criteria.')
print('Unique trials flagged:',len(flagged),'out of',len(T)); display(flagged[KEY+['session','n_flags']].head(20))
# Plot the six largest flex ranges, including the extreme that robust medians may hide.
review=T.nlargest(6,'flex_range_max_recomputed')
fig,axes=plt.subplots(3,2,figsize=(14,10))
for ax,(_,r) in zip(axes.flat,review.iterrows()):
    d=raw[(raw.participant==r.participant)&(raw.gesture_id==r.gesture_id)&(raw.trial_id==r.trial_id)]
    for c in FLEX: ax.plot(d.timestamp_ms,d[c]-d[c].median(),label=c.replace('flex_',''))
    ax.set_title(f'{r.participant} {r.gesture_id} trial {r.trial_id} ({r.session})'); ax.set_xlabel('ms'); ax.set_ylabel('ADC relative to trial median')
axes.flat[0].legend(fontsize=8); plot_save('manual_review_flex_traces')
# Saved list ranks review priorities without labeling trials bad.
save(review[KEY+['session','flex_range_max_recomputed','gyro_mag_max','orientation_diameter_deg']],
     'priority_manual_review.csv','Six largest flex-range traces for manual inspection; review candidates, not exclusions.')


## Phase 20 — Rejected and failed attempts kept separate

**A. Purpose.** Assess operator rejections and connection failures without mixing them into accepted data.

**C. Expected output.** Counts by participant/gesture/reason and a separate rejected-signal descriptive table.

**D. Interpretation.** Wrong gestures may be very steady, so motion QC alone cannot detect semantic labeling errors.

**E. Genuine problem.** Rejected/accepted exact-payload contamination would have failed phase 2.

**F. Warning only.** Connection failures can leave no complete CSV; no rows or summaries are invented for those attempts.

**G. Do not conclude.** Rejections are not additional training examples and cannot be used to estimate sensitivity of all collection QC.

**B. Runnable code:**


In [ ]:
reject_stats=[]
for p in sorted((ROOT/'raw').glob('*/rejected/*.csv')):
    m=re.fullmatch(r'(G\d+)_T(\d+)_A(\d+)_DISCARDED\.csv',p.name)
    d=units(cache[p]); pid=p.parent.parent.name
    log=logs[(logs.participant==pid)&(logs.data_file==str(p.relative_to(p.parent.parent)))].iloc[0]
    reject_stats.append(dict(participant=pid,gesture_id=m[1],trial_id=int(m[2]),attempt=int(m[3]),reason=log.reason,
          n_rows=len(d),gyro_mag_max=d.gyro_magnitude_dps.max(),flex_range_max_recomputed=d[FLEX].max().sub(d[FLEX].min()).max(),**q_metrics(d)))
reject_stats=pd.DataFrame(reject_stats)
save(reject_stats,'rejected_signal_statistics.csv','Rejected-only descriptive signals; explicitly excluded from T and PCA.')
save(logs.groupby(['participant','gesture_id','result','reason'],dropna=False).size().reset_index(name='attempts'),
     'attempt_detail_counts.csv','Accepted/discarded/failed counts by participant, gesture and reason.')
display(reject_stats)
compare=pd.concat([T[['gyro_mag_max','orientation_p95_deg','flex_range_max_recomputed']].assign(result='ACCEPTED'),
                   reject_stats[['gyro_mag_max','orientation_p95_deg','flex_range_max_recomputed']].assign(result='DISCARDED')])
display(compare.groupby('result').agg(['count','median','max']))


## Phase 21 — Evidence dashboard and scientific verdict

**A. Purpose.** Convert audit results into specific actions, rather than a generic claim of quality.

**C. Expected output.** A dashboard with PASS/REVIEW/FAIL, affected units, a 16-question narrative and saved output guide.

**D. Interpretation.** Hard audit errors determine FAIL. Empirical outliers, baseline change and uncertain orientation features determine REVIEW.

**E. Genuine problem.** Objective encoding, provenance or structural failures must be resolved before modeling.

**F. Warning only.** Review items may remain documented limitations after inspection. No data are automatically removed.

**G. Do not conclude.** This EDA cannot establish classifier accuracy, true label correctness, absolute orientation accuracy, or population generalization.

**B. Runnable code:**


In [ ]:
dashboard=[]
def dash(category,status,evidence,affected,action):
    dashboard.append(dict(category=category,status=status,evidence=evidence,affected_participants_trials=affected,recommended_next_action=action))
hard=sum(a['status']=='FAIL' for a in audit)
for cat,evidence in [('Structural completeness',f'{len(GFILES)} files; {len(T)} trials; {len(raw)} samples'),
 ('Participant balance',str(balance.sum(axis=1).to_dict())),('Gesture balance',str(balance.sum(axis=0).to_dict())),
 ('Rows/trial integrity',str(integrity.n_rows.value_counts().to_dict())),('Missing values',str(int(raw[SCHEMA].isna().sum().sum()))),
 ('Duplicates',f'{int(integrity.duplicate_rows.sum())} exact within-trial duplicates')]:
    dash(cat,'FAIL' if hard else 'PASS',evidence,'All accepted trials','Retain audited trial identities; resolve any audit FAIL.')
late_max=T.late_max_us.max(); missed=T.imu_missed.sum()
dash('Timing integrity','PASS' if integrity.timestamp_ok.all() and missed==0 and T.interval_min_ms.eq(40).all() and T.interval_max_ms.eq(40).all() else 'REVIEW',
     f'All timestamp grids match={integrity.timestamp_ok.all()}; logged interval range {T.interval_min_ms.min()}–{T.interval_max_ms.max()} ms; max lateness {late_max:g} us; missed IMU {missed:g}',
     'All accepted trials','CSV and firmware logs support scheduled timing; independent hardware timestamps/firmware needed to prove actual sensor timing.')
rail=int(rail_summary.exact_rail_samples.sum()); oob=int(rail_summary.outside_samples.sum())
dash('ADC saturation','FAIL' if oob else ('REVIEW' if rail else 'PASS'),f'{rail} exact rail samples; {oob} outside bounds',
     'See flex_sensor_summary.csv','Inspect rail/encoding issues if present; ADC-to-angle calibration not implied.')
flex_ids=T.loc[T.flex_range_max_recomputed.eq(T.flex_range_max_recomputed.max()),KEY].to_dict('records')
dash('Flex stability','REVIEW',f'Median largest-finger range={T.flex_range_max_recomputed.median():g}; maximum={T.flex_range_max_recomputed.max():g} ADC; constant-channel trials={sum(T[c+"_unique_values"].eq(1).sum() for c in FLEX)}',str(flex_ids),'Inspect priority_manual_review.csv and traces; retain raw signals.')
dash('IMU stability','REVIEW',f'Gyro peak median/p95/max={T.gyro_mag_max.median():.3f}/{T.gyro_mag_max.quantile(.95):.3f}/{T.gyro_mag_max.max():.3f} deg/s; acceleration median range={T.accel_mag_median.min():.4f}–{T.accel_mag_median.max():.4f} g',
     'qc_flagged_trials.csv','Inspect statistical extremes; uncorrected gyro includes bias.')
dash('Quaternion validity','PASS' if not (qnorm==0).any() else 'FAIL',f'Norm {qnorm.min():.8f}–{qnorm.max():.8f}; maximum norm error {qerr.max():.8g}', 'All accepted samples','Normalize copies for rotation calculations; retain norm-error flags.')
dash('Orientation stability','REVIEW',f'Pairwise diameter median/p95/max={T.orientation_diameter_deg.median():.3f}/{T.orientation_diameter_deg.quantile(.95):.3f}/{T.orientation_diameter_deg.max():.3f} degrees','qc_flagged_trials.csv','Review unusual holds; distinguish within-trial and between-session effects.')
dash('Calibration quality','REVIEW',f'{len(calmeta)} recorded calibrations; maximum neutral mean CSV-summary difference={cal.neutral_mean_difference.abs().max():.6g}; maximum neutral p95 angle={cal.orientation_p95_deg.max():.3f} deg','calibration_summary_eda.csv','Confirm summary definitions and timing relative to use; inspect calibration outliers.')
dash('Neutral-to-fist dynamic range','REVIEW',f'Absolute spans {cal.absolute_span.min():.3f}–{cal.absolute_span.max():.3f} ADC; signed range {cal.signed_span.min():.3f}–{cal.signed_span.max():.3f}', 'calibration_summary_eda.csv','Compare calibration representations in held-out participant validation; no assumed increase with bend.')
dash('Checkpoint drift','REVIEW',f'Neutral-return deviation {drift.loc[drift.kind=="checkpoint","drift_deg"].min():.3f}–{drift.loc[drift.kind=="checkpoint","drift_deg"].max():.3f} deg', 'checkpoint_drift.csv','Audit absolute orientation features; physical repositioning versus estimator drift is unresolved.')
dash('Final drift','REVIEW',f'{len(finaldrift)} final files; deviation {finaldrift.drift_deg.min():.3f}–{finaldrift.drift_deg.max():.3f} deg',str(finaldrift[['participant','session','accepted_trials_in_session']].to_dict('records')),'Document absent final check and zero-accepted-session final check; do not transfer across sessions.')
dash('Session/recalibration events','REVIEW',f'{len(session_summary)} sessions; {int(events.event.eq("RECALIBRATION_NEEDED").sum())} recalibration-needed events',str(session_summary[session_summary.accepted_trials.eq(0)][['participant','session']].to_dict('records')),'Retain reference, boot and epoch boundaries.')
dash('Connection failures','REVIEW',f'{int(logs.result.eq("FAILED").sum())} failed attempts; {int(events.event.eq("LINK_LOST").sum())} link-loss events','P02/P03/P05 per logs','Document acquisition interruptions; accepted payloads independently audited.')
dash('Rejected attempts','PASS',f'{int(logs.result.eq("DISCARDED").sum())} discarded recordings kept separate; no exact accepted payload copies detected', 'attempt_detail_counts.csv','Retain separately for provenance; never use as accepted examples.')
dash('Participant variability','REVIEW',f'Raw flex participant fractions: {variation.participant_fraction.min():.1%}–{variation.participant_fraction.max():.1%}; interaction: {variation.interaction_fraction.min():.1%}–{variation.interaction_fraction.max():.1%}', 'All 10 participants','Use participant-held-out outer evaluation and participant-grouped inner model selection.')
dash('Major anomalies','REVIEW',f'{len(irregularities)} documented irregularities; {len(flagged)} distinct trial review candidates', 'irregularities.csv and qc_flagged_trials.csv','Resolve metadata history, gesture descriptions and priority traces; no blanket exclusion.')
quality=pd.DataFrame(dashboard)
save(quality,'final_quality_summary.csv','Evidence/action dashboard. PASS is scoped to the stated evidence; REVIEW is not FAIL.'); display(quality)
verdict='DATA INTEGRITY PROBLEMS SHOULD BE RESOLVED FIRST' if hard or oob or flaglong.severity.eq('FAIL').any() else ('READY FOR PREPROCESSING, WITH SPECIFIC TRIALS/ISSUES TO REVIEW' if quality.status.eq('REVIEW').any() else 'READY FOR PREPROCESSING')
closest=pairs.iloc[0]
nscores=norm_sil[norm_sil.labels=='gesture_id'].set_index('representation').silhouette
thumb=cal[cal.finger=='flex_thumb']
thumb_review=cal_review[cal_review.finger=='flex_thumb'][['participant','session','signed_span']].to_dict('records')
temporal_summary=assoc.groupby('covariate').rho.agg(['median','min','max']).round(3).to_dict('index')

report=f"""# Scientific interpretation\n\nVerdict: **{verdict}**\n\n1. **Structurally complete? DIRECT EVIDENCE:** {len(T)} accepted trial records, {len(raw)} samples, {len(GFILES)} gesture files; every participant × gesture cell contains {sorted(map(int,balance.stack().unique()))} trials. The five-repetition protocol is met when the audit passes. Ten people, not 500 independent people, underpin generalization.\n\n2. **Corrupt/incomplete accepted trials? DIRECT EVIDENCE:** {int(integrity.missing.sum())} missing cells, {int(integrity.infinite.sum())} infinite cells, {int(integrity.duplicate_rows.sum())} within-trial duplicate rows, and {int(integrity.n_rows.ne(50).sum())} wrong-length trials. Accepted CSV/log keys are one-to-one. NO EVIDENCE: this cannot independently verify a participant performed the stated gesture.\n\n3. **Sampling timing? DIRECT EVIDENCE:** exact 0–1960 ms grids at 40 ms steps; logged min/max intervals {T.interval_min_ms.min():g}/{T.interval_max_ms.max():g} ms, zero-missed status={missed==0}, maximum lateness {late_max:g} microseconds. INDIRECT EVIDENCE: these records support the collection schedule. NO EVIDENCE that the CSV grid alone establishes hardware sampling timestamps or sample freshness.\n\n4. **Five flex channels consistent? DIRECT EVIDENCE:** raw limits and per-finger spans are tabulated. Signed spans range {cal.signed_span.min():.2f} to {cal.signed_span.max():.2f} ADC. Participant main-effect fractions range {variation.participant_fraction.min():.1%}–{variation.participant_fraction.max():.1%}; participant×gesture fractions range {variation.interaction_fraction.min():.1%}–{variation.interaction_fraction.max():.1%}. Different participant/finger response patterns remain important. Thumb reference warnings: {thumb_review}; opposite direction is not by itself proof of hardware polarity reversal.\n\n5. **Saturated, stuck or noisy? DIRECT EVIDENCE:** {rail} exact rail samples, {oob} out-of-bounds readings; see constant-channel counts in flex_sensor_summary.csv. Largest within-trial finger range is {T.flex_range_max_recomputed.max():g} ADC at {flex_ids}. INDIRECT EVIDENCE: unusually large steps/ranges merit review; no single cause is established. A flat static trace alone does not prove a stuck sensor.\n\n6. **Sufficiently stationary? DIRECT EVIDENCE:** gyro peak median/p95/max={T.gyro_mag_max.median():.3f}/{T.gyro_mag_max.quantile(.95):.3f}/{T.gyro_mag_max.max():.3f} deg/s; quaternion pairwise diameter median/p95/max={T.orientation_diameter_deg.median():.3f}/{T.orientation_diameter_deg.quantile(.95):.3f}/{T.orientation_diameter_deg.max():.3f} degrees. INDIRECT EVIDENCE: many holds show limited quaternion motion. NO EVIDENCE: a validated universal acceptable-motion boundary for this classifier; review joint flex/gyro/quaternion traces. Logged gyro peaks differ from recomputed raw peaks by up to {T.gyro_peak_log_difference.abs().max():.3f} deg/s; even bias correction leaves discrepancies. Inspect firmware calculation and sample rate before treating these as the same metric.\n\n7. **Quaternion validity? DIRECT EVIDENCE:** norms {qnorm.min():.8f}–{qnorm.max():.8f}, error mean {qerr.mean():.8g}, p95 {np.percentile(qerr,95):.8g}, max {qerr.max():.8g}. Angular calculations normalize nonzero quaternions and respect sign equivalence. NO EVIDENCE: normalization guarantees orientation accuracy.\n\n8. **Orientation drift? DIRECT EVIDENCE:** same-session neutral-return angles span {drift.drift_deg.min():.3f}–{drift.drift_deg.max():.3f} degrees over all checkpoint/final files. Maximum absolute logged-vs-recomputed difference is {drift.logged_minus_recomputed.abs().max():.4f} degrees. INDIRECT EVIDENCE: large deviations undermine assuming a common absolute orientation across collection. NO EVIDENCE: these deviations are purely yaw drift, or entirely electronic rather than neutral-pose repositioning.\n\n9. **Restarts and segmentation? DIRECT EVIDENCE:** accepted log entries match references, boot and epoch; {len(session_summary)} named sessions include unused calibrations. Inspect session_summary.csv. In this archive P05 final drift belongs to a session without accepted trials; P01 has extra R06 and no final-drift/completion event. INDIRECT EVIDENCE: preserved segmentation makes session-aware analysis possible, not proof that all post-restart features are comparable.\n\n10. **Inter-participant variation? DIRECT EVIDENCE:** inspect per-finger participant, gesture and interaction variance fractions and the same-feature PCA colored two ways. These are descriptive decompositions of this balanced sample. Temporal associations after within-session/gesture demeaning vary: {temporal_summary}. No uniform causal temperature effect is established. NO EVIDENCE: population representativeness or unseen-person accuracy.\n\n11. **Will calibration help? DIRECT EVIDENCE:** normalization_variation_comparison.csv and normalization_silhouette_comparison.csv compare the same eligible trials; {int(common.sum())}/{len(T)} qualify for the exploratory span rule. DIRECT EVIDENCE: gesture silhouette in five-flex space is {nscores.get('raw',float('nan')):.3f} raw, {nscores.get('neutral_subtracted',float('nan')):.3f} neutral-subtracted and {nscores.get('span_scaled',float('nan')):.3f} span-scaled. These results do not support assuming that calibration improves class geometry; inspect the thumb references first. INDIRECT EVIDENCE: calibration may introduce reference-posture or denominator problems. NO EVIDENCE: better held-out performance; no model was trained. Calibration after a recording cannot serve as a prospective baseline without an explicit deployment protocol.\n\n12. **Potentially difficult gesture pairs? DIRECT EVIDENCE:** the lowest raw-flex centroid/radius ratio is {closest.gesture_a} ({closest.name_a}) versus {closest.gesture_b} ({closest.name_b}), ratio {closest.separation_ratio:.3f}. Full pair rankings are saved. INDIRECT EVIDENCE: these pairs may deserve feature/trace investigation. NO EVIDENCE: predicted confusion rates.\n\n13. **Which exact trials need inspection? DIRECT EVIDENCE:** {len(flagged)} distinct accepted trials receive at least one exploratory flag. qc_flagged_trials.csv is the complete exact list; qc_flags_long.csv contains each metric and numerical threshold. priority_manual_review.csv and the six trace plots start with largest flex ranges. The review count is not a count of bad trials.\n\n14. **Exclude anything before preprocessing?** Structural evidence does not justify blanket deletion of accepted trials when the hard audit passes. Keep the nine discarded recordings and failed attempts out of accepted data by their existing status. Confirm any accepted-trial exclusion by documented trace/protocol evidence, with sensitivity analysis. NO EVIDENCE: an empirical outlier by itself is invalid.\n\n15. **Document versus correct?** Correct ambiguous metadata only after checking collection history; retain originals and a change log. Complete gesture-definition placeholders. Document retries, calibration rejection events, P01 checkpoint/completion irregularities and calibration-only sessions. Do not fabricate final checks or merge orientation chains across references.\n\n16. **Next preprocessing experiments:** preserve one row per trial and participant grouping; compare robust flex medians alone versus variability summaries, neutral-subtracted and signed-span flex, and flex plus gravity/acceleration features. Audit absolute quaternion/Euler features before use because long-session deviation is large. Investigate isolated flex steps before choosing smoothing or clipping. Fit scaling/feature selection on training participants only. Use a participant-held-out outer loop and participant-grouped inner loop for tuning; if unseen-person calibration will be available, simulate its exact timing and labeled-reference availability. Report per-participant variability, not just pooled performance. No KNN or final classifier is trained here.\n\n**{verdict}**\n"""
(OUT/'SCIENTIFIC_INTERPRETATION.md').write_text(report)
OUTPUT_DESCRIPTIONS['SCIENTIFIC_INTERPRETATION.md']='Sixteen evidence-qualified answers and justified verdict from the executed data.'
display(Markdown(report))
readme='SMARTGLOVE EDA OUTPUT GUIDE\n\nStart with SCIENTIFIC_INTERPRETATION.md, final_quality_summary.csv and priority_manual_review.csv.\nPASS is scoped to the specific check; REVIEW does not mean delete. Never treat sample rows as independent trials.\n\n'
readme+='\n'.join(f'{name}: {meaning}' for name,meaning in OUTPUT_DESCRIPTIONS.items())
readme+='\n\nplots/: question-driven figures.\nextracted_input/: unchanged source bytes when a ZIP was supplied; not newly cleaned data.\n'
readme+='\nUnits: ADC raw counts; gyro deg/s from /16.4; acceleration g from /8192; angles degrees. Configuration is supplied by the researcher, not established by the CSV.\n'
readme+='\nAll 3-IQR, near-rail and calibration denominator rules are exploratory review heuristics. No automated removal; raw features remain separate from calibration comparisons.\n'
(OUT/'EDA_RESULTS_README.txt').write_text(readme)
print(verdict)


## Sources and method notes

- Espressif, [ESP32 ADC programming guide](https://docs.espressif.com/projects/esp-idf/en/v4.2/esp32/api-reference/peripherals/adc.html): a 12-bit result has a maximum count of 4095. This supports the encoding check, not a guarantee of calibrated ADC voltage or bending angle.

- Markley, Cheng, Crassidis & Oshman (2007), *Averaging Quaternions*, Journal of Guidance, Control, and Dynamics 30(4), 1193–1197, [DOI 10.2514/1.28949](https://doi.org/10.2514/1.28949). [NASA author manuscript](https://ntrs.nasa.gov/citations/20070017872). This notebook uses the dominant eigenvector of the quaternion outer-product matrix; it is sign invariant.
- Dehghani et al. (2019), *A Quantitative Comparison of Overlapping and Non-Overlapping Sliding Windows for Human Activity Recognition Using Inertial Sensors*, Sensors 19(22), 5026, [DOI 10.3390/s19225026](https://doi.org/10.3390/s19225026). Relevant to evaluation dependence and the need to distinguish subject-dependent from subject-independent validation. The present data have repeated trials even without overlapping windows.
- Hardware conversion values and ADC encoding are **researcher-supplied acquisition settings**, not values inferred from signals. The MPU manufacturer documentation link was not retrievable during preparation, so this notebook does not claim independent firmware/register verification. Confirm the collection code/registers before publication.

The statistics, QC fences, signed-span comparisons and decomposition choices are defined explicitly in code. No external source supplies a universal good/bad threshold for your glove experiment. Dataset-specific conclusions come from the attached archive and its logs, not web sources.
